# Embeddings And Similarity Search

An embedding represents text as a numeric vector, placed so that texts
with similar meaning get nearby vectors. A retrieval system embeds stored
chunks and a question into the same space, then ranks chunks whose vectors
are close to the question vector. This notebook uses an intentionally
transparent local hash embedding so rankings can be inspected without a
network request.

## Learning Objectives

By the end of this notebook, you should be able to:

- Explain dense vectors and cosine similarity.
- Implement a deterministic local embedding baseline with NumPy.
- Perform exact top-k retrieval by exhaustively scoring every chunk.
- Diagnose where lexical features fail to represent meaning.

## From Text To Nearby Points

Similar texts should map to nearby vector locations. In learned models,
semantic relations emerge from training; in this transparent baseline,
token hashing and a small synonym map make the mechanics visible. It is
useful for learning and tests, not a substitute for a trained embedding
model.

<div align="center">
  <img src="assets/02-word-vector-space.png" alt="Semantic vector space relationships" width="400" style="max-width: 100%; height: auto;" />
  <p><em>Semantic vector space relationships. Source: <a href="https://commons.wikimedia.org/wiki/File:Word_vector_illustration.jpg" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>

Lexical (sparse) retrieval matches the words themselves: each dimension
stands for a word, and most entries are zero. A dense embedding is a
shorter, learned vector in which most entries are nonzero and no single
dimension has a meaning of its own; meaning is spread across all of them,
as the illustration below shows. Because they are learned, dense
embeddings can represent paraphrases even when the exact words differ.
That capability is why they are commonly used for RAG.

<div align="center">
  <img src="assets/02-dense-vector.png" alt="Sentence represented as a dense vector" width="600" style="max-width: 100%; height: auto;" />
  <p><em>Sentence represented as a dense vector. Source: <a href="https://qdrant.tech/articles/what-is-a-vector-database/" target="_blank" rel="noopener noreferrer">Qdrant</a></em></p>
</div>


## Similarity Search Flow

```mermaid
graph LR
    A["Chunks"] --> B["Embedding model"]
    B --> C["Normalized vectors"]
    Q["Question"] --> B
    C --> D["Cosine scores"]
    D --> E["Top-k chunks with metadata"]
```

Vectors are normalized to unit length, so a plain dot product gives their
cosine similarity: ranking then depends on the direction of the vectors,
not their magnitude.

The search returns the **top-k** chunks, the k chunks with the highest
scores. This notebook compares the question with every stored chunk
(**exhaustive scoring**) and then keeps the k best scores, which makes it
an **exact search**: the results are guaranteed to be the true top-k. The
approximate indexes in notebook 03 skip some chunks to save time, so they
may miss a true match.


In [ ]:
import hashlib
import json
import re
from pathlib import Path
from typing import Any

import numpy as np

# Keep the evidence collection fixed while changing only retrieval mechanics.
documents = json.loads((Path("data") / "course_handbook.json").read_text())
print(f"Loaded {len(documents)} source documents.")

Each compact source policy is used as one initial chunk. Notebook 01 explained why that is a defensible baseline for this corpus.


In [ ]:
chunks = [
    {
        "chunk_id": f"{document['doc_id']}-whole",
        "doc_id": document["doc_id"],
        "title": document["title"],
        "section": document["section"],
        "text": document["text"],
    }
    for document in documents
]
print(chunks[0])

## Build A Transparent Embedding Baseline

A hash embedding counts each token in a stable numeric dimension. A small
synonym map first rewrites known synonyms such as `reimbursement` to
`refund`, so both land in the same dimension. Each text uses only a few
dozen of the 2048 dimensions, so these vectors are sparse: the baseline is
a lexical retriever in vector form. Different tokens occasionally hash to
the same dimension (a collision) and become indistinguishable; 2048
dimensions keep this rare for a corpus of about 200 distinct words. Unlike
Python's process-randomized `hash()`, `hashlib.sha256` ensures that
re-running the notebook produces the same vector dimensions.


In [ ]:
SYNONYMS = {
    "reimbursement": "refund",
    "reimburse": "refund",
    "charged": "payment",
    "money": "payment",
    "transfer": "payment",
    "bill": "invoice",
    "duplicated": "duplicate",
    "login": "password",
    "signin": "password",
    "phone": "device",
    "factor": "authentication",
    "throttle": "limit",
    "throttled": "limit",
    "slow": "timeout",
    "lagging": "timeout",
    "fails": "timeout",
    "failing": "timeout",
    "inference": "prediction",
    "additional": "extension",
    "thirty": "30",
    "mentoring": "mentor",
    "file": "document",
    "show": "reveal",
}
STOPWORDS = {
    "a",
    "an",
    "and",
    "are",
    "can",
    "do",
    "for",
    "from",
    "how",
    "i",
    "in",
    "is",
    "my",
    "of",
    "should",
    "the",
    "to",
    "what",
    "when",
    "with",
}

Tokenization now applies that visible vocabulary policy before vector construction. Keeping it separate makes it easy to inspect which semantic shortcuts the baseline is allowed to use.


In [ ]:
def tokenize(text: str) -> list[str]:
    """Lowercase words, drop stopwords, and map a few domain synonyms."""

    # Synonym mapping lets obvious synonyms share a vector dimension.
    words: list[str] = re.findall(r"[a-z0-9]+", text.lower())
    return [SYNONYMS.get(word, word) for word in words if word not in STOPWORDS]

In [ ]:
class HashEmbeddingModel:
    """Create inspectable sparse hash vectors without downloading a model."""

    def __init__(self, dimensions: int = 2048) -> None:
        self.dimensions = dimensions

    def encode(self, texts: list[str]) -> np.ndarray:
        # One row per text keeps document and query vectors in the same space.
        vectors = np.zeros((len(texts), self.dimensions), dtype="float32")
        for row, text in enumerate(texts):
            for token in tokenize(text):
                # A stable hash makes token placement reproducible across runs.
                digest = hashlib.sha256(token.encode()).digest()
                index = int.from_bytes(digest[:4], "little") % self.dimensions
                vectors[row, index] += 1.0
        # Unit-length vectors turn dot product into cosine similarity.
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        return vectors / np.maximum(norms, 1e-12)

In production, this interface could be replaced with Sentence Transformers or a hosted embedding API without changing the retrieval contract: `encode(texts)` still returns vectors whose ranking should be evaluated.


In [ ]:
# Use enough dimensions that results are not driven by hash collisions.
embedder = HashEmbeddingModel(dimensions=2048)
chunk_vectors = embedder.encode([chunk["text"] for chunk in chunks])
print("Vector matrix shape:", chunk_vectors.shape)
print("First vector norm:", round(float(np.linalg.norm(chunk_vectors[0])), 4))

## Cosine Similarity

Cosine similarity is the cosine of the angle between two vectors: `1` means
they point the same way, `0` means they are perpendicular and share nothing,
and `-1` means they point in opposite directions. Because this notebook's
vectors contain only non-negative counts, its scores fall between 0 and 1.
Once vectors are normalized to unit length,
`query_vector @ document_vector` is the cosine score:

$$\operatorname{cosine}(q, d) = \frac{q \cdot d}{\lVert q \rVert \lVert d \rVert}$$

<br>
<div align="center">
  <img src="assets/02-cosine-similarity.png" alt="Cosine similarity geometry" width="450" style="max-width: 100%; height: auto;" />
  <p><em>Cosine similarity geometry. Source: <a href="https://commons.wikimedia.org/wiki/File:CosineSimilarity.png" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>


In [ ]:
def search(query: str, top_k: int = 3) -> list[dict[str, Any]]:
    """Rank every stored chunk by normalized-vector dot product."""

    query_vector = embedder.encode([query])[0]
    # Exhaustive scoring compares the query with every stored chunk vector.
    scores = chunk_vectors @ query_vector
    # Sorting all scores and keeping the first k makes the result exact.
    ranked_ids = np.argsort(scores)[::-1][:top_k]
    return [
        {**chunks[index], "score": round(float(scores[index]), 4)}
        for index in ranked_ids
    ]

In [ ]:
query = "I paid twice and need reimbursement for the duplicate invoice."
for result in search(query):
    print(result["doc_id"], result["score"], "-", result["title"])

The expected `refunds` policy should rank first because `reimbursement`
maps to `refund` and the question contains payment/invoice vocabulary. A
rank is evidence about this vectorizer and this corpus; it is not proof
that the generated answer will be correct.


## Inspect Several User Questions

Retrieval design should be judged across distinct intents rather than one convenient query.


In [ ]:
questions = [
    "How do I replace my multi-factor authentication after changing phone?",
    "The API returned 429, when should my client retry?",
    "Can I paste a user's email address into an external model prompt?",
    "How can I request additional time for my project?",
]

for question in questions:
    winner = search(question, top_k=1)[0]
    print(question, "=>", winner["doc_id"], winner["score"])

## Failure Analysis: Similarity Is Not Understanding

Hash features miss conceptual links unless words overlap or we manually
add synonym rules. Learned dense embedding models usually handle more
paraphrases, but they can still retrieve irrelevant text, encode bias, or
fail on domain-specific terminology. Evaluation remains mandatory.


In [ ]:
hard_questions = [
    "Someone duplicated a transfer, can money be returned?",
    "My second factor is inaccessible since I changed hardware.",
    "The inference service is lagging until the call fails.",
    "The scoring endpoint hangs before returning anything.",
]

for question in hard_questions:
    results = search(question, top_k=2)
    print("\n", question)
    for result in results:
        print(" ", result["doc_id"], result["score"])

The first three questions are recovered through the synonym map. The
scoring-endpoint phrasing remains a useful miss: it shares the
intended meaning without sufficient visible vocabulary. A low or incorrect
rank motivates a trained embedding model or additional evaluation
coverage. Do not expand synonym rules endlessly and mistake a hand-tuned
synonym list for robust semantic retrieval: mapping `returned` to `refund`,
for example, would also turn "The API returned 429" into a refund question.


## Scores Need A Confidence Policy

A top result is always returned, even for a question the corpus does not
answer. Applications should inspect score strength and the margin between
the first and second result rather than turning every nearest neighbor into
an answer. Thresholds are corpus- and embedding-specific; they must be
tuned with supported and unsupported examples.


In [ ]:
confidence_questions = [
    "How should I reset a forgotten password?",
    "Where can I park a bicycle near the campus?",
    "What recipe should I cook tonight?",
]

for question in confidence_questions:
    first, second = search(question, top_k=2)
    # A narrow top-two margin signals uncertainty even when one item ranks first.
    margin = first["score"] - second["score"]
    print("\n", question)
    print(
        f" top = {first['doc_id']}, score = {first['score']}, margin = {round(margin, 4)}"
    )

Unsupported questions may still overlap accidentally with a source or tie
at zero. The bicycle and recipe questions share no words with any policy,
so every score is 0.0: the reported top document is not a match at all,
just whichever chunk comes first when the tied scores are sorted. A robust
RAG pipeline combines retrieval thresholds, a prompt that permits
`I do not know`, and answer-grounding evaluation rather than using the top
score as truth. Notebook 05 measures retrieval quality separately.


## Metadata Enables Better Retrieval Products

Vector similarity is only one part of retrieval. Metadata lets an
application filter by policy section, cite the original page, implement
access rules, and remove outdated records. There are two strategies:
**post-filtering** ranks first and then removes disallowed results, while
**pre-filtering** restricts the search to allowed records before ranking.
The helper below post-filters by section; vector databases can often
pre-filter inside the search request.


In [ ]:
def search_in_section(query: str, section: str, top_k: int = 2) -> list[dict[str, Any]]:
    """Return scored results only from an allowed handbook section."""

    all_scores = chunk_vectors @ embedder.encode([query])[0]
    # Post-filter here for clarity; databases can pre-filter during search.
    allowed = [
        (index, float(score))
        for index, score in enumerate(all_scores)
        if chunks[index]["section"] == section
    ]
    allowed.sort(key=lambda pair: pair[1], reverse=True)
    return [
        {**chunks[index], "score": round(score, 4)} for index, score in allowed[:top_k]
    ]


for result in search_in_section("I paid twice for one invoice.", "Billing"):
    print(result["doc_id"], result["score"], "-", result["title"])

In a real service, post-filtering only a few retrieved candidates can
incorrectly remove all useful results. Pre-filter during retrieval, or
retrieve a sufficiently broad candidate pool before post-filtering, then
evaluate the combined policy.


## Exercise: Retrieve A Support Answer Source

Use `search()` to retrieve two sources for:

`My password link expired before I could open it.`

Check whether `password-reset` appears at rank 1 and print the supporting
text that would be passed to a generator.


In [ ]:
# @TODO: Retrieve two sources for the query and check which document ranks first.


### Reference Solution


In [ ]:
exercise_query = "My password link expired before I could open it."
exercise_results = search(exercise_query, top_k=2)
# Rank-one correctness is the simplest useful check for this known question.
correct_at_one = exercise_results[0]["doc_id"] == "password-reset"
assert correct_at_one, "The reset policy should rank first for this question."

print("Relevant result at rank 1:", correct_at_one)
for rank, result in enumerate(exercise_results, start=1):
    print(f"\nRank {rank}: {result['doc_id']} ({result['score']})")
    print(result["text"])

`password-reset` should appear first because it
contains both the reset-link action and its expiry condition. Supplying
that full chunk to generation preserves the instruction to request a new
link rather than inventing a recovery process.


## Next Step

This notebook scores every chunk vector. That exhaustive scan is
transparent but expensive at large scale. Next, FAISS will provide
optimized exact and approximate index structures while retaining
chunk metadata outside the index.

## Further Reading

- Sentence Transformers: [Docs](https://sbert.net/) / [Semantic Search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)
- NumPy: [Linear Algebra Functions](https://numpy.org/doc/stable/reference/routines.linalg.html)
- OpenAI: [Vector Embeddings](https://platform.openai.com/docs/guides/embeddings)
- [Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks](https://arxiv.org/abs/1908.10084) - Reimers and Gurevych, 2019
